# Importing libraries and data

In [ ]:
import foldersfiles as ff
import functions as fn

import numpy as np
import pandas as pd

import pickle

In [ ]:
!ls "{ff.CLEANED_ORIGINALS_DATA_DIR / ff.PARQUETS}"

In [ ]:
!ls "{ff.CLEANED_PROCESSED_DATA_DIR / ff.PICKLES}"

In [ ]:
df = pd.read_parquet(ff.CLEANED_ORIGINALS_DATA_DIR / ff.PARQUETS / f'{ff.ICONS_RICOV}.parquet')

# with open(ff.CLEANED_ORIGINALS_DATA_DIR / ff.PICKLES / f'{ff.}.pickle', 'rb') as file:
# 	df = pickle.load(file)

# Overview

In [ ]:
df.info()

In [ ]:
df

# Analysis

## Hospitalizations

In [ ]:
df_dates = df[['pseudopaz', 'data_apertura', 'data_chiusura', 'data_chiusura_sdo']]

# df_dates = df_dates.sort_values(['pseudo_paz', 'data_apertura']).reset_index(drop=true)

In [ ]:
# Check the difference between the 'data_chiusura' and the 'data_chiusura_sdo' fields

df_dates['diff_data_chiusura'] = df_dates['data_chiusura'] - df_dates['data_chiusura_sdo']
df_dates['diff_data_chiusura']
df_dates['diff_data_chiusura'].describe()
# df_dates['data_chiusura_sdo']
# diff_no_nat = df_dates['diff_data_chiusura'].dropna()
# diff_no_nat.describe()

In [ ]:
# # 1) check that data_apertura is before data_chiusura for every line
# df_dates['check_date_order'] = df_dates['data_apertura'] < df_dates['data_chiusura']
# print(df_dates['check_date_order'].value_counts())

# # rows where the check fails
# invalid_rows = df_dates[~df_dates['check_date_order']]
# invalid_rows

# 2) compute the duration of each stay
df_dates['durata'] = df_dates['data_chiusura'] - df_dates['data_apertura']

# aggregate per pseudopatient
summary = df_dates.groupby('pseudopaz')['durata'].agg(['count', 'sum', 'mean'])
summary.columns = ['n_ricoveri', 'tempo_totale', 'tempo_medio']

# round to the nearest day
summary['tempo_totale'] = summary['tempo_totale'].dt.round('D')
summary['tempo_medio'] = summary['tempo_medio'].dt.round('D')

summary = summary.reset_index()

summary

## Hospital codes

In [ ]:
region_map = {
    '042001': 'trento',
    '042004': 'trento',
    '042005': 'trento',
    '042007': 'trento',
    '042014': 'trento',
    '042015': 'trento',
    '042022': 'trento',
    '042006': 'rovereto',
    '042010': 'rovereto',
    '042016': 'rovereto',
    '042018': 'rovereto',
    '042019': 'rovereto',
    '042020': 'rovereto',
}

In [ ]:
hospital_map = {
    '042001': 'tn schiara',
    '042004': 'b_valsugana',
    '042005': 'cles',
    '042006': 'rovereto',
    '042007': 'tione',
    '042010': 'arco',
    '042014': 'cavalese',
    '042015': 'tn scamillo',
    '042016': 's.stefano/s.pancrazio',
    '042018': 'cdc eremo_arco',
    '042019': 'cdc regina',
    '042020': 'cdc solatrix',
    '042022': 'cdc villa_bianca',
}

# hospital_map = {
#     '042001': 'OSPEDALE DI TRENTO',
#     '042004': 'PRESIDIO OSPEDALIERO DI BORGO VALSUGANA',
#     '042005': 'PRESIDIO OSPEDALIERO DI CLES',
#     '042006': 'OSPEDALE DI ROVERETO',
#     '042007': 'PRESIDIO OSPEDALIERO DI TIONE',
#     '042010': 'PRESIDIO OSPEDALIERO DI ARCO',
#     '042014': 'PRESIDIO OSPEDALIERO DI CAVALESE',
#     '042015': 'OSPEDALE SAN CAMILLO',
#     '042016': 'IST.RIAB.S.STEFANO-U.O. OSP. S.PANCRAZIO',
#     '042018': 'CASA DI CURA EREMO DI ARCO SRL',
#     '042019': 'CASA DI CURA REGINA',
#     '042020': 'CASA DI CURA SOLATRIX',
#     '042022': 'CASA DI CURA VILLA BIANCA',
# }

In [ ]:
df_strutture = df[['pseudopaz', 'struttura_cod']]

# df_strutture = df_strutture.sort_values(['pseudo_paz', 'data_apertura']).reset_index(drop=true)

In [ ]:
df_strutture['regione'] = df_strutture['struttura_cod'].map(region_map)

region_counts = df_strutture.groupby('pseudopaz')['regione'].nunique()

same_region = region_counts[region_counts == 1].index
different_region = region_counts[region_counts > 1].index

print('Patients always in same region:', len(same_region))
print('Patients across different regions:', len(different_region))

In [ ]:
df_strutture['ospedale'] = df_strutture['struttura_cod'].map(hospital_map)

switched_df_strutture = df_strutture[df_strutture['pseudopaz'].isin(different_region)]

regions = switched_df_strutture.groupby('pseudopaz').agg(
    codice=('struttura_cod', 'unique'),
    visitati=('ospedale', 'unique')
).reset_index()

regions.insert(1, 'num', regions['codice'].apply(len))

regions['codice'] = regions['codice'].apply(lambda x: ', '.join(x))
regions['visitati'] = regions['visitati'].apply(lambda x: ', '.join(x))

regions = regions.sort_values('num', ascending=False)

regions = regions.reset_index(drop=True)

regions

## Export

In [ ]:
# fn.export_func(df_, ff.CLEANED_PROCESSED_DATA_DIR, '', filetype=[
#     # 'parquet' ,
#     # 'pickle' ,
#     'excel'
# ])